In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import os
import sys
from typing import Any

import lightning as L
import torch
from dotenv import load_dotenv
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.loggers import MLFlowLogger
from loguru import logger
from mlflow.models.signature import infer_signature
from pydantic import BaseModel, PrivateAttr
from torch.utils.data import DataLoader
import pandas as pd
import mlflow

sys.path.insert(0, "..")

from src.utils.embedding_id_mapper import IDMapper
from src.skipgram.dataset import SkipGramDataset
# from src.skipgram.inference import SkipGramInferenceWrapper
from src.skipgram.model import SkipGram
from src.skipgram.trainer import LitSkipGram
from src.utils.embedding_id_mapper import IDMapper

_ = load_dotenv(override = True)

## Controller

In [3]:
class Args(BaseModel):
    testing: bool = False
    log_to_mlflow: bool = True
    _mlf_logger: Any = PrivateAttr()
    experiment_name: str = "RecSys MVP - Item2Vec"
    run_name: str = "000-first-attempt"
    notebook_persist_dp: str = None
    random_seed: int = 41
    device: str = None

    max_epochs: int = 100
    batch_size: int = 128

    num_negative_samples: int = 2
    window_size: int = 1

    embedding_dim: int = 128
    early_stopping_patience: int = 15
    learning_rate: float = 0.01
    l2_reg: float = 1e-5

    mlf_model_name: str = "item2vec"
    min_roc_auc: float = 0.7

    train_data_path: str = "../data/interim/train.parquet"
    val_data_path:str = "../data/interim/val.parquet"
    test_data_path:str = "../data/interim/test.parquet"

    def init(self):
        self.notebook_persist_dp = os.path.abspath(f"data/011-item2vec/{self.run_name}")
        os.makedirs(self.notebook_persist_dp, exist_ok=True)

        if not (mlflow_uri := os.environ.get("MLFLOW_TRACKING_URI")):
            logger.warning(
                f"Environment variable MLFLOW_TRACKING_URI is not set. Setting self.log_to_mlflow to false."
            )
            self.log_to_mlflow = False

        if self.log_to_mlflow:
            logger.info(
                f"Setting up MLflow experiment {self.experiment_name} - run {self.run_name}..."
            )
            self._mlf_logger = MLFlowLogger(
                experiment_name=self.experiment_name,
                run_name=self.run_name,
                tracking_uri=mlflow_uri,
                log_model=True,
            )

        return self


args = Args().init()

print(args.model_dump_json(indent=2))

2025-02-04 10:49:41.823 | INFO     | __main__:init:40 - Setting up MLflow experiment RecSys MVP - Item2Vec - run 000-first-attempt...
{
  "testing": false,
  "log_to_mlflow": true,
  "experiment_name": "RecSys MVP - Item2Vec",
  "run_name": "000-first-attempt",
  "notebook_persist_dp": "/home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/notebooks/data/011-item2vec/000-first-attempt",
  "random_seed": 41,
  "device": null,
  "max_epochs": 100,
  "batch_size": 128,
  "num_negative_samples": 2,
  "window_size": 1,
  "embedding_dim": 128,
  "early_stopping_patience": 15,
  "learning_rate": 0.01,
  "l2_reg": 0.00001,
  "mlf_model_name": "item2vec",
  "min_roc_auc": 0.7,
  "train_data_path": "../data/interim/train.parquet",
  "val_data_path": "../data/interim/val.parquet",
  "test_data_path": "../data/interim/test.parquet"
}


In [4]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
# device = 'cpu'
logger.info(f"Using {device} device")

2025-02-04 10:49:47.314 | INFO     | __main__:<module>:7 - Using cuda device


In [5]:
def init_model(n_items, embedding_dim, device):
    model = SkipGram(n_items, embedding_dim).to(device)
    return model

# Test implementation

In [15]:
n_items = 1000
window_size = 1
negative_samples = 2
batch_size = 2

model = init_model(n_items, args.embedding_dim, device = "cpu")

# Example inputs
target_items = torch.tensor([1, 2, 3, 1000])  # Target item IDs
context_items = torch.tensor([10, 20, 30, 40])  # Context item IDs
labels = torch.tensor([1, 0, 1])  # Positive or negative context pairs

predictions = model(target_items, context_items)
print(predictions)

tensor([0.4896, 0.4920, 0.4953, 0.4982], grad_fn=<SigmoidBackward0>)


# Prep data

In [16]:
# Read data
train_df = pd.read_parquet(args.train_data_path)
val_df = pd.read_parquet(args.val_data_path)

In [17]:
train_df.head(1)

,customer_id,event_timestamp,article_id
393,a36ab8958f1eb3e65bd2299316b0c8a17a70eb5fbbbd52...,2020-03-02,372860002


In [18]:
val_df.head(1)

,customer_id,event_timestamp,article_id
50209,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,2020-07-23,823505002


In [19]:
idm = IDMapper().load("../data/idm.json")

print(len(idm.item_to_index))

train_dataset = SkipGramDataset(
    train_interaction_df= train_df, 
    val_interaction_df= None,
    mode = "train",
    item_id_to_idx=idm.item_to_index,
)
val_dataset = SkipGramDataset(
    train_interaction_df= train_df, 
    val_interaction_df= val_df,
    mode = "val",
    item_id_to_idx=idm.item_to_index,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    drop_last=True,
    collate_fn=train_dataset.collate_fn,
    num_workers=4,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    drop_last=True,
    collate_fn=val_dataset.collate_fn,
)


645
2025-02-04 10:52:28.382 | INFO     | src.skipgram.dataset:__init__:67 - Processing sequences...
2025-02-04 10:52:28.403 | INFO     | src.skipgram.dataset:__init__:67 - Processing sequences...


In [20]:
assert train_dataset.item_id_to_idx == idm.item_to_index, "ID Mappings are not matched!"

# Train

In [114]:
n_items = len(train_dataset.items)
n_items

645

#### Optinal: Overfit 1 batch

In [115]:
batch_size = 1  # Need to set to 1 else can not learn
embedding_dim = 32
window_size = 1
num_negative_samples = 2

batch_dataset = SkipGramDataset(
    train_interaction_df= train_df[0:10], 
    val_interaction_df= None,
    mode = "train",
    negative_samples= num_negative_samples,
    window_size= window_size,
)

batch_dataloader = DataLoader(
    batch_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    collate_fn=batch_dataset.collate_fn,
)


model = init_model(len(batch_dataset.items), embedding_dim, device)

2025-01-24 12:30:19.580 | INFO     | src.skipgram.dataset:__init__:67 - Processing sequences...


In [116]:
early_stopping = EarlyStopping(
    monitor="val_loss", patience=10, mode="min", verbose=False
)

# model
model = init_model(len(batch_dataset.items), args.embedding_dim, device= device)
lit_model = LitSkipGram(
    model, learning_rate=0.01, l2_reg=0.0, log_dir=args.notebook_persist_dp
)

log_dir = f"{args.notebook_persist_dp}/logs/overfit"

# train model
trainer = L.Trainer(
    default_root_dir=log_dir,
    max_epochs=100,
    overfit_batches=1,
    callbacks=[early_stopping],
)
trainer.fit(
    model=lit_model,
    train_dataloaders=batch_dataloader,
    val_dataloaders=batch_dataloader,
)
logger.info(f"Logs available at {trainer.log_dir}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(overfit_batches=1)` was configured so 1 batch will be used.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type     | Params | Mode 
----------------------------------------------------
0 | skipgram_model | SkipGram | 640    | train
----------------------------------------------------
640       Trainable params
0         Non-trainable params
640       Total params
0.003     Total estimated model params size (MB)
2         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:252: UserWarning:

You requested to overfit but enabled val dataloader shuffling. We are turning off the val dataloader shuffling for you.

/home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: PossibleUserWarning:

The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

/home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:252: UserWarning:

You requested to overfit but enabled train dataloader shuffling. We are turning off the train dataloader shuffling for you.

/home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/.venv/lib/python3.

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


2025-01-24 12:30:22.130 | INFO     | __main__:<module>:25 - Logs available at /home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/notebooks/data/011-item2vec/000-first-attempt/logs/overfit/lightning_logs/version_10


In [117]:
model(torch.tensor([0]), torch.tensor([3]))

tensor([0.0002], grad_fn=<SigmoidBackward0>)

# Run with all data

In [118]:
# papermill_description=fit-model
early_stopping = EarlyStopping(
    monitor="val_loss", patience=args.early_stopping_patience, mode="min", verbose=False
)

checkpoint_callback = ModelCheckpoint(
    dirpath=f"{args.notebook_persist_dp}/checkpoints",
    filename="best-checkpoint",
    save_top_k=1,
    monitor="val_loss",
    mode="min",
)

# model
model = init_model(n_items, args.embedding_dim, device=device)
lit_model = LitSkipGram(
    model,
    learning_rate=args.learning_rate,
    l2_reg=args.l2_reg,
    log_dir=args.notebook_persist_dp,
)

log_dir = f"{args.notebook_persist_dp}/logs/run"

# train model
trainer = L.Trainer(
    default_root_dir=log_dir,
    max_epochs=args.max_epochs,
    callbacks=[early_stopping, checkpoint_callback],
    accelerator=args.device if args.device else "auto",
    logger=args._mlf_logger if args.log_to_mlflow else None,
)
trainer.fit(model=lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/.venv/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: UserWarning:

Checkpoint directory /home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/notebooks/data/011-item2vec/000-first-attempt/checkpoints exists and is not empty.

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type     | Params | Mode 
----------------------------------------------------
0 | skipgram_model | SkipGram | 82.7 K | train
----------------------------------------------------
82.7 K    Trainable params
0         Non-trainable params
82.7 K    Total params
0.331     Total estimated model params size (MB)
2         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: PossibleUserWarning:

The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.



Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

🏃 View run 000-first-attempt at: http://localhost:5002/#/experiments/3/runs/d9ffa207c9e24227898a8fa189de7db0
🧪 View experiment at: http://localhost:5002/#/experiments/3


In [119]:
best_trainer = LitSkipGram.load_from_checkpoint(
    checkpoint_callback.best_model_path,
    skipgram_model=init_model(n_items, args.embedding_dim, device="cpu"),
)

In [120]:
best_model = best_trainer.skipgram_model
best_model.eval()

SkipGram(
  (embeddings): Embedding(646, 128, padding_idx=645)
)

In [133]:
best_model.to("cpu").embeddings(torch.tensor([0]))[: ,: 10]


tensor([[ 0.1321, -0.0645, -0.2666, -0.3447,  0.1405,  0.3136,  0.2233,  0.6977,
          0.1289, -0.2502]], grad_fn=<SliceBackward0>)

In [134]:
best_model(torch.tensor([600]), torch.tensor([603]))

tensor([0.8226], grad_fn=<SigmoidBackward0>)

## Register model

In [ ]:
from src.skipgram.inference import SkipGramInferenceWrapper
inferrer = SkipGramInferenceWrapper(best_model)

In [137]:
sample_input = {
    "item_1_ids": [train_dataset.item_idx_to_id[0]],
    "item_2_ids": [train_dataset.item_idx_to_id[1]],
}
sample_output = inferrer.infer([0], [1])
sample_output

array([0.42892772], dtype=float32)

In [156]:
if args.log_to_mlflow:
    run_id = trainer.logger.run_id
    sample_output_np = sample_output
    signature = infer_signature(sample_input, sample_output_np)
    with mlflow.start_run(run_id=run_id):
        mlflow.pyfunc.log_model(
            python_model=inferrer,
            artifact_path="inferrer",
            # We log the id_mapping to the predict function so that it can accept item_id and automatically convert ot item_indice for PyTorch model to use
            artifacts={"id_mapping": "/home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/data/idm.json"},
            signature=signature,
            input_example=sample_input,
            registered_model_name=args.mlf_model_name,
        )

2025/01/24 16:47:32 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
Successfully registered model 'item2vec'.
2025/01/24 16:47:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: item2vec, version 1


🏃 View run 000-first-attempt at: http://localhost:5002/#/experiments/3/runs/d9ffa207c9e24227898a8fa189de7db0
🧪 View experiment at: http://localhost:5002/#/experiments/3


Created version '1' of model 'item2vec'.


In [158]:
# Set the new model as champion
mlf_client = trainer.logger.experiment
if args.log_to_mlflow:
    val_roc_auc = trainer.logger.experiment.get_run(trainer.logger.run_id).data.metrics[
        "val_roc_auc"
    ]

    if val_roc_auc > args.min_roc_auc:
        logger.info(f"Aliasing the new model as champion...")
        model_version = (
            mlf_client.get_registered_model(args.mlf_model_name)
            .latest_versions[0]
            .version
        )

        mlf_client.set_registered_model_alias(
            name=args.mlf_model_name, alias="champion", version=model_version
        )

        mlf_client.set_model_version_tag(
            name=args.mlf_model_name,
            version=model_version,
            key="author",
            value="dinhnehehe",
        )

2025-01-24 16:50:09.916 | INFO     | __main__:<module>:9 - Aliasing the new model as champion...


In [23]:
model_uri = f"models:/{args.mlf_model_name}@champion"
model = mlflow.pyfunc.load_model(model_uri)

In [28]:
sample_input = {
    "item_1_ids": ["1111111111"],
    "item_2_ids": ["2222"],
}
model.predict(sample_input)


{'item_1_ids': ['1111111111'], 'item_2_ids': ['2222'], 'scores': [0.5]}

In [29]:
sample_input = {
    "item_1_ids": [train_dataset.item_idx_to_id[2]],
    "item_2_ids": [train_dataset.item_idx_to_id[1]],
}
model.predict(sample_input)

{'item_1_ids': ['160442010'],
 'item_2_ids': ['160442007'],
 'scores': [0.9116064310073853]}

## Log params and clean up

In [ ]:
all_params = [args]

if args.log_to_mlflow:
    with mlflow.start_run(run_id=run_id):
        for params in all_params:
            params_dict = params.model_dump()
            params_ = {
                f"{params.__repr_name__()}.{k}": v for k, v in params_dict.items()
            }
            mlflow.log_params(params_)